# P01 · Radiografía de un mercado financiero
**QuantLab | Módulo 1 · Fundamentos financieros**

## 1. Objetivo
Comprobar que nuestro entorno funciona y familiarizarnos con el Bid, Ask y último precio de BTC/EUR. Este cuaderno es el punto de partida, no una estrategia de inversión.

## 2. Configuración
En Colab selecciona CPU. En local utiliza el entorno `.venv` y el kernel QuantLab. Ejecuta todas las celdas en orden. Si faltan dependencias en Colab, ejecuta `%pip install requests pandas numpy` en una celda aparte.

In [1]:
import sys
import platform
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from pathlib import Path

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p01.quotes import SYNTHETIC_QUOTE, fetch_kraken_ticker, quote_metrics, round_trip_cost

print("Python:", sys.version.split()[0])
print("Sistema:", platform.system())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Requests:", requests.__version__)

Python: 3.13.5
Sistema: Linux
Pandas: 2.2.3
NumPy: 2.3.5
Requests: 2.32.5


## 3. Datos y procedencia
Por defecto utilizamos un ejemplo **ficticio**, de modo que la práctica es reproducible sin conexión. Cuando quieras consultar datos reales, cambia `USE_LIVE_DATA` a `True` y vuelve a ejecutar la celda. La hora de recepción no es la hora del evento del exchange.

Fuente para el modo real: [documentación del ticker público de Kraken](https://docs.kraken.com/api/docs/rest-api/get-ticker-information/).

In [2]:
USE_LIVE_DATA = False

# El código de la consulta está en src/quantlab/p01/quotes.py (fetch_kraken_ticker)
market = fetch_kraken_ticker() if USE_LIVE_DATA else dict(SYNTHETIC_QUOTE)
pd.DataFrame([market]).T.rename(columns={0: "Valor"})

,Valor
source,synthetic_example
received_at,None
symbol,BTC/EUR
best_bid,59998.0
best_ask,60002.0
last_price,59999.0


## 4. Interpretación financiera
- **Bid:** mejor precio comprador disponible.
- **Ask:** mejor precio vendedor disponible.
- **Last price:** precio de la última operación ejecutada; no tiene por qué coincidir con la cotización actual.

Una compra inmediata de pequeño tamaño cruza el Ask y una venta inmediata cruza el Bid, si hay profundidad suficiente.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿los datos tienen sentido?</b>

Antes de calcular nada comprobamos que la cotización es coherente. Si alguna condición falla, la celda **se detiene con un error** en vez de seguir con datos malos.

- `best_bid > 0` y `last_price > 0`: un precio negativo o cero indica un dato corrupto.
- `best_ask >= best_bid`: si el Ask fuera menor que el Bid, el libro estaría **cruzado** (alguien compraría más caro de lo que otro vende y la operación ya se habría ejecutado). Suele indicar datos desincronizados o un error de lectura.

</div>

In [3]:
# 🔎 FASE DE VALIDACIÓN: si algo falla, se detiene aquí
assert market["best_bid"] > 0
assert market["best_ask"] >= market["best_bid"]
assert market["last_price"] > 0
print("Comprobaciones básicas superadas.")

Comprobaciones básicas superadas.


<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Validación</b>

Los datos superan las comprobaciones mínimas: precios positivos y libro **no cruzado** (Ask ≥ Bid). Pasar estas pruebas **no garantiza** que el dato sea correcto o reciente; solo descarta errores evidentes.

📄 Ver `docs/conceptos/validacion_de_datos.md`

</div>

## 5. Ejercicio pendiente
Implementa una función que calcule `mid_price`, `spread`, `relative_spread`, `spread_bps` y `last_price_deviation`. Usa las lecciones 1.1 y 1.2 como referencia. Con el ejemplo ficticio, comprueba que el mid es 60.000 EUR y el spread 4 EUR.

¿Qué precio utilizarías para comprar inmediatamente? ¿Y para vender? ¿Cuál sería el coste implícito de una compra y venta de 0,01 BTC, sin comisiones y con precios constantes?

### Solución del ejercicio 5
La función `quote_metrics()` está en **`src/quantlab/p01/quotes.py`** y sus pruebas en **`tests/test_quotes.py`** (`pytest` desde la raíz del proyecto).

`last_price_deviation` no está definida en las lecciones 1.1 y 1.2. Aquí la definimos como la distancia del último precio al mid: `last_price − mid`, en EUR y en bps del mid.

In [ ]:
import inspect
print(inspect.getsource(quote_metrics))  # muestra el código de src/ para estudiarlo

metrics = quote_metrics(market["best_bid"], market["best_ask"], market["last_price"])
pd.Series(metrics, name="Valor").to_frame()

In [ ]:
# 🔎 FASE DE VALIDACIÓN: comprobamos la función con el ejemplo conocido
if market["source"] == "synthetic_example":
    assert metrics["mid_price"] == 60_000.0
    assert metrics["spread"] == 4.0
    assert abs(metrics["spread_bps"] - 0.6667) < 1e-3
    assert metrics["last_price_deviation"] == -1.0
    print("Ejemplo ficticio verificado: mid = 60.000 EUR y spread = 4 EUR.")

In [ ]:
qty = 0.01  # BTC
buy_cost = qty * market["best_ask"]     # comprar ya: cruzas el Ask
sell_income = qty * market["best_bid"]  # vender ya: cruzas el Bid
rt_cost = round_trip_cost(market["best_bid"], market["best_ask"], qty)  # = qty * spread

print(f"Compra inmediata de {qty} BTC al Ask {market['best_ask']:,.2f} EUR -> pagas {buy_cost:,.2f} EUR")
print(f"Venta inmediata de {qty} BTC al Bid {market['best_bid']:,.2f} EUR -> recibes {sell_income:,.2f} EUR")
print(f"Coste implícito de ida y vuelta: {rt_cost:.2f} EUR "
      f"({rt_cost / (qty * metrics['mid_price']) * 10_000:.3f} bps del nominal en mid)")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Precios de ejecución y coste implícito</b>

- **Comprar inmediatamente → Ask** (60.002 €): el mejor precio al que alguien vende.
- **Vender inmediatamente → Bid** (59.998 €): el mejor precio al que alguien compra.
- **Ida y vuelta de 0,01 BTC:** pagas 600,02 € y recibes 599,98 € → **pierdes 0,04 €** = spread × cantidad (4 € × 0,01) ≈ **0,67 bps**. Cada pata cuesta **medio spread** frente al mid.
- Supuestos: sin comisiones, precios constantes y profundidad suficiente en el primer nivel.

📄 Ver `docs/conceptos/spread_y_costes.md`

</div>

### Comparación: ejemplo ficticio vs. Kraken en vivo
Cambia `COMPARE_LIVE` a `True` para consultar el ticker real (necesita Internet). Por defecto está desactivado para que el cuaderno se pueda ejecutar sin conexión.

In [ ]:
COMPARE_LIVE = False

if COMPARE_LIVE:
    live = fetch_kraken_ticker()
    rows = {}
    for name, q in {"Ficticio": SYNTHETIC_QUOTE, "Kraken (vivo)": live}.items():
        m = quote_metrics(q["best_bid"], q["best_ask"], q["last_price"])
        rows[name] = {
            "bid": q["best_bid"], "ask": q["best_ask"], "last": q["last_price"],
            "mid": m["mid_price"], "spread_eur": m["spread"], "spread_bps": m["spread_bps"],
            "last_dev_eur": m["last_price_deviation"],
            "coste_0.01_btc_eur": round_trip_cost(q["best_bid"], q["best_ask"], 0.01),
        }
    print("Recibido:", live["received_at"])
    display(pd.DataFrame(rows).round(4))
else:
    print("Comparación en vivo desactivada (COMPARE_LIVE = False).")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Ficticio vs. vivo (19-11-2026, dos consultas)</b>

| | Bid | Ask | Spread | Spread bps | Last − Mid | Coste 0,01 BTC |
|---|---|---|---|---|---|---|
| Ficticio | 59.998,0 € | 60.002,0 € | 4,0 € | 0,667 | −1,0 € | 0,04 € |
| Kraken (1.ª) | 57.967,8 € | 57.970,8 € | 3,0 € | 0,518 | −2,2 € | 0,030 € |
| Kraken (2.ª, 09:01 UTC) | 57.960,3 € | 57.961,2 € | 0,9 € | 0,155 | +0,45 € | 0,009 € |

- El ejemplo ficticio es **realista**: el spread real de BTC/EUR en Kraken tiene el mismo orden de magnitud (≈ 0,5 bps).
- BTC/EUR es un mercado **muy líquido**: comprar y vender al instante cuesta menos de 1 bps. Las comisiones del exchange (no incluidas) pesarán mucho más que el spread.
- El last quedó una vez **por debajo** del mid (cerca del Bid) y otra **por encima** (igual al Ask). Depende de si la última operación fue una venta o una compra agresiva: es historia, no una oferta actual.
- **El spread cambia mucho en minutos**: de 3,0 € a 0,9 € entre dos consultas. No hay que generalizar a partir de una foto; hacen falta series temporales.

📄 Ver `docs/conclusiones/P01_market_snapshot.md`

</div>

## 6. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P01</b>

1. **El entorno funciona:** el kernel QuantLab ejecuta el cuaderno de principio a fin.
2. **No existe "un" precio:** hay un precio para comprar (Ask), otro para vender (Bid) y una referencia calculada (Mid) que **no es ejecutable**.
3. **El Last Price es historia:** es la última operación, no una oferta disponible ahora (aquí está 1 € por debajo del mid).
4. **Operar cuesta el spread:** comprar y vender al instante pierde el spread completo aunque el precio no se mueva.
5. **Los bps permiten comparar:** 4 € de spread en BTC ≈ 0,67 bps; la misma cifra en euros pesaría mucho más en un activo barato.
6. **Validar antes de calcular:** un libro cruzado o precios ≤ 0 deben detener el análisis.
7. **Con una sola observación no hay señal predictiva.** Necesitaremos series temporales.

📄 Resumen en `docs/conclusiones/P01_market_snapshot.md`

</div>